# Analyse du dédoublonnage

## Fonction d'analyse
Calcul et restitution du nombre de ligne en défut d'intégrité

In [1]:
from datetime import datetime
import json
from tab_dataset import Cdataset
import pandas as pd
import ntv_pandas as npd
import pathlib

def analyse_integrite(data, schema, affiche=True, indic=True):
    '''analyse les relations du DataFrame 'data' définies dans le schéma 'schema'.
    Le nombre de lignes en erreur par relation (dict) est retourné et optionnellement affiché (paramètre 'affiche=True') . 
    Les lignes en erreur sont optionnellement ajoutées (paramètre 'indic=True') à 'data' sous forme de champs booléens par relation.
    '''
    dic_errors = Cdataset(data).check_relationship(schema)
    dic_count = {name: len(errors) for name, errors in dic_errors.items()}
    if affiche:
        for name, total in dic_count.items():
            print('{:<50} {:>5}'.format(name, total))
    if indic:
        data['ok'] = True
        for name, errors in dic_errors.items():
            data[name] = True
            data.loc[errors, name] = False
            data['ok'] = data['ok'] & data[name] 
        if affiche:
            nb_ok = sum(data['ok'])
            nb_ko = len(data) - sum(data['ok'])      
            print("\nnombre d'enregistrements sans erreurs : ", nb_ok)
            print("nombre d'enregistrements avec au moins une erreur : ", nb_ko)
            print("dont doublons : ", dic_count['index - id_pdc_itinerance'])
            print("\ntaux d'erreur : ", round(nb_ko / len(data) * 100), ' %')
    return dic_count

## Schéma de données
Le schéma de données restreint à la propriété 'relationship' et construit à partir du modèle de données est le suivants :

In [2]:
# complément à inclure dans le schéma de données
schema = {
    'relationships': [
         # relation unicité des pdl
         {"fields": ["id_pdc_itinerance", "index"],                    "link" : "coupled" },   
         # relations inter entités
         {"fields": ["id_station_itinerance", "contact_operateur"],    "link" : "derived" },
         {"fields": ["id_station_itinerance", "nom_enseigne"],         "link" : "derived" },
         {"fields": ["id_station_itinerance", "coordonneesXY"],        "link" : "derived" },
         {"fields": ["id_pdc_itinerance", "id_station_itinerance"],    "link" : "derived" },
         # relations intra entité - station
         {"fields": ["id_station_itinerance", "nom_station"],          "link" : "derived" },
         {"fields": ["id_station_itinerance", "implantation_station"], "link" : "derived" },
         #{"fields": ["id_station_itinerance", "date_maj"],             "link" : "derived" },
         {"fields": ["id_station_itinerance", "nbre_pdc"],             "link" : "derived" },
         {"fields": ["id_station_itinerance", "condition_acces"],      "link" : "derived" },
         {"fields": ["id_station_itinerance", "horaires"],             "link" : "derived" },
         {"fields": ["id_station_itinerance", "station_deux_roues"],   "link" : "derived" },
         # relations intra entité - localisation
         {"fields": ["coordonneesXY", "adresse_station"],              "link" : "derived" }
    ]
}

## Initialisation des données
Fichier pandas

In [3]:
file_irve_brut = 'consolidation-etalab-schema-irve-statique-v-2.3.1-20260301.csv'
file_irve = 'consolidation-etalab-schema-irve-statique-v-2.3.1-20260401.csv'

irve_brut = pd.read_csv(file_irve_brut, sep=',', low_memory=False, dtype='object').reset_index()
irve_brut['last_modified'] = irve_brut['datagouv_last_modified']

irve = pd.read_csv(file_irve, sep=',', low_memory=False, dtype='object').reset_index()
irve['last_modified'] = irve['datagouv_last_modified']

print('nombre de lignes : ', len(irve_brut), len(irve))

nombre de lignes :  379946 148286


## Bilan d'intégrité
Qualification du dédoublonnage

In [4]:
resultat = analyse_integrite(irve, schema)

index - id_pdc_itinerance                            816
contact_operateur - id_station_itinerance           1777
nom_enseigne - id_station_itinerance                4064
coordonneesXY - id_station_itinerance               5402
id_station_itinerance - id_pdc_itinerance             28
nom_station - id_station_itinerance                 2436
implantation_station - id_station_itinerance        2408
nbre_pdc - id_station_itinerance                    5166
condition_acces - id_station_itinerance              280
horaires - id_station_itinerance                    1079
station_deux_roues - id_station_itinerance            73
adresse_station - coordonneesXY                     8224

nombre d'enregistrements sans erreurs :  132857
nombre d'enregistrements avec au moins une erreur :  15429
dont doublons :  816

taux d'erreur :  10  %


## Analyse

In [5]:
irve_brut.groupby(['datagouv_organization_or_owner']).count()['index'].sort_values(ascending=False)[0:10]

datagouv_organization_or_owner
QualiCharge                       61545
Engie Mobilités Electriques       58305
GIREVE                            31714
TotalEnergies Marketing France    29490
Mobilize Power Solutions          21033
IZIVIA                            15767
Driveco                           15551
ubitricity                        15349
Alizé                             14334
STATIONS-E                        13260
Name: index, dtype: int64

In [6]:
irve.groupby(['datagouv_organization_or_owner']).count()['index'].sort_values(ascending=False)[0:10]

datagouv_organization_or_owner
QualiCharge                                  46503
GIREVE                                       31714
Alizé                                        13624
IZIVIA                                       12958
Eco-Movement                                 10806
Indigo Group                                  6907
Driveco                                       3935
TotalEnergies Marketing France                1849
Engie Mobilités Electriques                   1827
Citeos Ingénierie IdF & Est (Cogelum IdF)     1485
Name: index, dtype: int64

In [7]:
irve.groupby(['datagouv_organization_or_owner', 'ok']).count()['index'].sort_values(ascending=False)[0:20]

datagouv_organization_or_owner             ok   
QualiCharge                                True     41357
GIREVE                                     True     29117
IZIVIA                                     True     11858
Alizé                                      True     11808
Eco-Movement                               True      9970
Indigo Group                               True      6861
QualiCharge                                False     5146
Driveco                                    True      3841
GIREVE                                     False     2597
Alizé                                      False     1816
TotalEnergies Marketing France             True      1730
Engie Mobilités Electriques                True      1581
Citeos Ingénierie IdF & Est (Cogelum IdF)  True      1465
Electric 55 Charging                       True      1291
Qovoltis                                   True      1186
IZIVIA                                     False     1100
e-Totem                

## Dédoublonnage proposé

In [8]:
id_stations = ['id_station_itinerance', 'date_maj', 'datagouv_organization_or_owner']
att_stations = ['last_modified', 'nom_station', 'adresse_station', 'coordonnéeXY']
stations = irve_brut.drop_duplicates(id_stations).copy()
id_stations_itinerance = irve_brut.drop_duplicates('id_station_itinerance').copy()
len(stations), len(id_stations_itinerance)

(79474, 59045)

### Dédoublonnage direct station

In [9]:
stations['qualicharge'] = stations['datagouv_organization_or_owner'] == 'QualiCharge'
stat_direct = stations.sort_values(by=['id_station_itinerance', 'qualicharge', 'date_maj', 'last_modified']).drop_duplicates('id_station_itinerance', keep='last').copy()

stations['qualicharge'].sum(), stat_direct['qualicharge'].sum(), len(stat_direct)

(13597, 13597, 59045)

In [10]:
stat_direct.groupby(['datagouv_organization_or_owner']).count()['index'].sort_values(ascending=False)[0:15]

datagouv_organization_or_owner
QualiCharge                                  13597
Eco-Movement                                 10485
GIREVE                                        9873
IZIVIA                                        6656
Alizé                                         4483
GREENEA                                       1910
Load Stations                                 1091
Driveco                                        892
Syndicat Départemental d'Energie du Tarn       700
Citeos Ingénierie IdF & Est (Cogelum IdF)      616
ZE-WATT                                        607
SOREGIES                                       451
Electric 55 Charging                           391
ZEborne                                        377
Mobilize Power Solutions                       373
Name: index, dtype: int64

### Dédoublonnage direct pdc

In [11]:
dedoubl = stat_direct[['id_station_itinerance', 'datagouv_organization_or_owner', 'date_maj',  'last_modified', 'qualicharge']].merge(irve_brut, how='left', on=['id_station_itinerance', 'datagouv_organization_or_owner', 'date_maj',  'last_modified'])
dedoubl_unique = dedoubl.sort_values(by=['id_station_itinerance', 'id_pdc_itinerance', 'qualicharge', 'date_maj', 'last_modified']).drop_duplicates(['id_station_itinerance', 'id_pdc_itinerance'], keep='last').copy()
pdc_direct =  dedoubl_unique.sort_values(by=['id_pdc_itinerance', 'qualicharge', 'date_maj', 'last_modified']).drop_duplicates('id_pdc_itinerance', keep='last').copy()
del(pdc_direct['index'])
pdc_direct = pdc_direct.reset_index()

len(dedoubl), len(dedoubl_unique), len(pdc_direct)

(173088, 172428, 144768)

In [12]:
pdc_direct.groupby(['datagouv_organization_or_owner']).count()['index'].sort_values(ascending=False)[0:10]

datagouv_organization_or_owner
QualiCharge                       61545
GIREVE                            22430
Alizé                             13576
IZIVIA                            12241
Indigo Group                       6907
Eco-Movement                       3912
Driveco                            3460
TotalEnergies Marketing France     1848
Engie Mobilités Electriques        1745
Electric 55 Charging               1291
Name: index, dtype: int64

In [13]:
resultat = analyse_integrite(pdc_direct, schema)

index - id_pdc_itinerance                              0
contact_operateur - id_station_itinerance              2
nom_enseigne - id_station_itinerance                   7
coordonneesXY - id_station_itinerance                964
id_station_itinerance - id_pdc_itinerance              0
nom_station - id_station_itinerance                 1246
implantation_station - id_station_itinerance          54
nbre_pdc - id_station_itinerance                    1144
condition_acces - id_station_itinerance               23
horaires - id_station_itinerance                     102
station_deux_roues - id_station_itinerance             0
adresse_station - coordonneesXY                     7670

nombre d'enregistrements sans erreurs :  135263
nombre d'enregistrements avec au moins une erreur :  9505
dont doublons :  0

taux d'erreur :  7  %


### Dédoublonnage indirect

In [14]:
stat_direct['dupl_coord_owner'] = ~stat_direct.duplicated(keep=False, subset=['coordonneesXY', 'datagouv_organization_or_owner'])
sum(stat_direct['dupl_coord_owner'])

33990

In [15]:
stat_direct[stat_direct['dupl_coord_owner']][['id_station_itinerance', 'id_pdc_itinerance', 'coordonneesXY', 'datagouv_organization_or_owner']].sort_values(by=['coordonneesXY'])

,id_station_itinerance,id_pdc_itinerance,coordonneesXY,datagouv_organization_or_owner
347372,FRS14PLLXJ0T1CUX9Q4H,FRS14ERHZJ2,"[-0.00097, 49.32456]",QualiCharge
250280,FRRVEP2143879729942805269,FRRVEECUSDE65LUZ1651201CP1,"[-0.00165, 42.87351]",GIREVE
229744,FRS65E65295001,FRS65E652950011,"[-0.00165, 42.87351]",Alizé
338451,FRSRGP90289192,FRSRGE12346396701,"[-0.00364, 46.66112]",QualiCharge
224414,FRS65E65168001,FRS65E651680011,"[-0.00492, 42.87247]",Alizé
...,...,...,...,...
243244,FRFR1P6405335469425843518,FRFR1ELQGD1,"[9.53079, 42.40091]",GIREVE
257373,FRLE2P1304745373839139761,FRLE2EAHCB1,"[9.54738, 42.26291]",GIREVE
260735,FRFR1P7983351901844776075,FRFR1ELGSD2,"[9.55017, 42.11461]",GIREVE
260805,FRFR1P6805277750442739212,FRFR1EDLVC1,"[9.55227, 42.16075]",GIREVE


In [16]:
stat_xy = stations.sort_values(by=['coordonneesXY', 'qualicharge', 'date_maj', 'last_modified']).drop_duplicates('id_station_itinerance', keep='last').copy()
stations['qualicharge'].sum(), stat_xy['qualicharge'].sum(), len(stat_xy)

(13597, 10574, 59045)